# Time Series Activity Forecasting Process

This notebook outlines a detailed process for forecasting activity (e.g., "seen" events) for various indicators over time using a combination of feature engineering, statistical models, machine learning, and ensemble methods. Below is a step-by-step description of the workflow:

---

## 1. Data Loading and Preparation

- **Data Source:** The data is loaded from a CSV file containing time series records for multiple indicators.
- **Preprocessing:**
    - The `date` column is parsed as a datetime object.
    - Data is sorted by `indicator` and `date`.
    - Only the most recent `n_days` (e.g., 100) are retained for analysis.
    - The resulting DataFrame (`df`) contains columns such as `API_UserName`, `date`, `indicator`, `observations`, `dayofweek`, `is_weekend`, `day`, `month`, and `seen`.

---

## 2. Feature Engineering

- **Time Series Features:** For each indicator, the following features are extracted:
    - `last_seen`: Days since the indicator was last seen.
    - `freq_7`, `freq_30`: Number of times seen in the last 7 and 30 days.
    - `avg_gap`: Average gap between seen events.
    - `burstiness`: Variability in the gaps between events.
    - `label_7`, `label_14`, `label_30`: Binary labels indicating if the indicator was seen in the last 7, 14, or 30 days.
- **Output:** A features DataFrame (`features_df`) indexed by indicator.

---

## 3. Model Training and Prediction

- **Models Used:**
    - **Logistic Regression:** Predicts the probability of being seen in the next 7, 14, or 30 days.
    - **Gradient Boosted Trees (GBT):** Another probabilistic classifier for the same task.
    - **Exponential Model:** Uses recent frequency to estimate the probability of being seen.
    - **Weibull AFT Model:** Survival analysis model to estimate the probability of future events.
- **Predictions:** For each indicator, probabilities are computed for being seen today, in 7, 14, and 30 days.

---

## 4. Rule-Based and Ensemble Methods

- **Rule-Based Labels:** Simple heuristics based on `last_seen` (e.g., if last seen within 7 days, label as active).
- **Logistic Model on Rules:** Trains a logistic regression model using the rule-based labels.
- **Ensemble:** Combines probabilities from the logistic model, GBT, Weibull, and exponential models using weighted averages to improve robustness.

---

## 5. Confidence Assessment and Formatting

- **Confidence Labels:** For each forecast window (today, 7d, 14d, 30d), assigns a qualitative confidence label (e.g., "Highly likely", "Possibly active", "Low confidence") based on probability thresholds and recent frequency.
- **Formatting:** Probabilities are formatted as percentages for presentation.

---

## Summary

This process enables robust, interpretable, and actionable forecasting of indicator activity using a blend of statistical and machine learning techniques, with clear outputs for operational use.

In [112]:
import numpy as np
import pandas as pd
import os
import datetime
from datetime import datetime
from datetime import timedelta
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from lifelines import WeibullAFTFitter
import warnings

In [113]:
from datetime import datetime, timedelta

# Set start_full_date_str and start_date_str to 100 days before today
today = datetime.today()
start_full_date_str = (today - timedelta(days=100)).strftime("%Y-%m-%d")
start_date_str = start_full_date_str
# Define the file path template
file_path_template = "Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"


In [114]:
# Calculate today's date
today = datetime.today()
print(today)

# Calculate end date (today + 0 days)
end_dt = today + timedelta(days=0)
end_date_str = end_dt.strftime("%Y-%m-%d")
print(end_date_str)

# Convert string dates to datetime objects
start_full_dt = datetime.strptime(start_full_date_str, "%Y-%m-%d")
start_dt = datetime.strptime(start_date_str, "%Y-%m-%d")

2025-06-16 09:23:24.615999
2025-06-16


In [115]:
# Function to load files and save them to the specified directory
def load_files(filenames):
    dataframes = []
    for filename in filenames:
        if not os.path.exists(filename):
            print(f"File {filename} does not exist. Skipping.")
            continue
        df = pd.read_csv(filename)
        dataframes.append(df)
    return dataframes


# Define the file path template and date range
date_format = "%Y%m%d"
start_date = datetime.strptime(start_date_str, "%Y-%m-%d")  # Convert start_date_str to datetime
dates_to_pull = pd.date_range(start_date, end_dt, freq='D')

# Generate the list of file paths
datelist = [file_path_template.format(date=dt.strftime(date_format)) for dt in dates_to_pull]
print(datelist)

# Concatenate dataframes from the list of files
src = pd.concat(load_files(datelist), ignore_index=True)

# Clean up the 'indicator' and 'OpDiv' columns if they exist
if 'indicator' in src.columns:
    src['indicator'] = src['indicator'].astype(str).str.split(' ', expand=True)[0].str.strip()
if 'OpDiv' in src.columns:
    src['OpDiv'] = src['OpDiv'].astype(str).str.strip()

# Display the cleaned dataframe
display(src)

['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250308.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250309.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250310.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250311.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250312.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250313.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250314.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250315.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250316.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250317.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250318.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250319.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/hto

,indicator,API_UserName,obs_date,OpDiv,indicator_key,observations,curr_date
0,102.90.61.13,64853912235916233429,2025-03-08,OS,102.90.61.13_OS,6800,2025-03-08
1,102.91.94.193,64853912235916233429,2025-03-08,OS,102.91.94.193_OS,5751,2025-03-08
2,103.225.136.166,64853912235916233429,2025-03-08,OS,103.225.136.166_OS,10,2025-03-08
3,104.18.68.40,15920309593055310684,2025-03-08,CMS,104.18.68.40_CMS,4,2025-03-08
4,104.18.69.40,15920309593055310684,2025-03-08,CMS,104.18.69.40_CMS,8,2025-03-08
...,...,...,...,...,...,...,...
68789,jendolstores.com,50189120947314147395,2025-06-16,NIH,jendolstores.com_NIH,1,2025-06-16
68790,104.234.115.167,74198686107399967946,2025-06-16,VA,104.234.115.167_VA,1,2025-06-16
68791,alsumood@emirates.net.ae,13983566077459384977,2025-06-16,FDA,alsumood@emirates.net.ae_FDA,1,2025-06-16
68792,167.89.109.112,13983566077459384977,2025-06-16,FDA,167.89.109.112_FDA,1,2025-06-16


In [116]:
src.drop(columns=['curr_date', 'indicator_key'], inplace=True)
src.rename(columns={'obs_date': 'date'}, inplace=True)
src['date'] = pd.to_datetime(src['date'])
src.reset_index(drop=True, inplace=True)


In [117]:
src

,indicator,API_UserName,date,OpDiv,observations
0,102.90.61.13,64853912235916233429,2025-03-08,OS,6800
1,102.91.94.193,64853912235916233429,2025-03-08,OS,5751
2,103.225.136.166,64853912235916233429,2025-03-08,OS,10
3,104.18.68.40,15920309593055310684,2025-03-08,CMS,4
4,104.18.69.40,15920309593055310684,2025-03-08,CMS,8
...,...,...,...,...,...
68789,jendolstores.com,50189120947314147395,2025-06-16,NIH,1
68790,104.234.115.167,74198686107399967946,2025-06-16,VA,1
68791,alsumood@emirates.net.ae,13983566077459384977,2025-06-16,FDA,1
68792,167.89.109.112,13983566077459384977,2025-06-16,FDA,1


In [118]:
src[src['indicator'] == '192.124.249.112']

,indicator,API_UserName,date,OpDiv,observations
14310,192.124.249.112,50189120947314147395,2025-04-15,NIH,4
14999,192.124.249.112,80363974983666420473,2025-04-16,IHS,13
15000,192.124.249.112,50189120947314147395,2025-04-16,NIH,151
15001,192.124.249.112,74198686107399967946,2025-04-16,VA,14
15738,192.124.249.112,00818860012482918321,2025-04-17,CDC,7
...,...,...,...,...,...
65327,192.124.249.112,74198686107399967946,2025-06-14,VA,21
66733,192.124.249.112,80363974983666420473,2025-06-15,IHS,1
66734,192.124.249.112,50189120947314147395,2025-06-15,NIH,14
66735,192.124.249.112,74198686107399967946,2025-06-15,VA,4


In [119]:
# Group the src dataframe by 'OpDiv' and get a dictionary of DataFrames for each OpDiv
opdiv_groups = {opdiv: group for opdiv, group in src.groupby('OpDiv')}

# Allow searching by indicator
def get_by_indicator(indicator_value):
    return src[src['indicator'] == indicator_value]

# Example usage:
get_by_indicator('192.124.249.112')

,indicator,API_UserName,date,OpDiv,observations
14310,192.124.249.112,50189120947314147395,2025-04-15,NIH,4
14999,192.124.249.112,80363974983666420473,2025-04-16,IHS,13
15000,192.124.249.112,50189120947314147395,2025-04-16,NIH,151
15001,192.124.249.112,74198686107399967946,2025-04-16,VA,14
15738,192.124.249.112,00818860012482918321,2025-04-17,CDC,7
...,...,...,...,...,...
65327,192.124.249.112,74198686107399967946,2025-06-14,VA,21
66733,192.124.249.112,80363974983666420473,2025-06-15,IHS,1
66734,192.124.249.112,50189120947314147395,2025-06-15,NIH,14
66735,192.124.249.112,74198686107399967946,2025-06-15,VA,4


In [120]:
opdiv_merged = {}

for opdiv, group_df in opdiv_groups.items():
    group_df['date'] = pd.to_datetime(group_df['date'])
    all_users = group_df['API_UserName'].unique()
    all_indicators = group_df['indicator'].unique()
    all_dates = pd.date_range(start=group_df['date'].min(), end=pd.Timestamp.now().normalize(), freq='D')
    all_combinations = pd.MultiIndex.from_product(
        [all_users, all_dates, all_indicators],
        names=['API_UserName', 'date', 'indicator']
    ).to_frame(index=False)
    all_combinations['OpDiv'] = opdiv  # Add OpDiv column

    merged = all_combinations.merge(group_df, how='left', on=['API_UserName', 'date', 'indicator', 'OpDiv'])
    merged['observations'] = merged['observations'].fillna(0).astype(int)
    merged['date'] = pd.to_datetime(merged['date'])
    merged['dayofweek'] = merged['date'].dt.dayofweek
    merged['is_weekend'] = merged['dayofweek'].isin([5, 6])
    merged['day'] = merged['date'].dt.day
    merged['month'] = merged['date'].dt.month
    merged['seen'] = (merged['observations'] > 0).astype(int)
    opdiv_merged[opdiv] = merged

# Example: display the merged dataframe for one OpDiv
display(opdiv_merged['DHA'])


,API_UserName,date,indicator,OpDiv,observations,dayofweek,is_weekend,day,month,seen
0,20790633968691748718,2025-03-26,104.160.6.2,DHA,2,2,False,26,3,1
1,20790633968691748718,2025-03-26,141.98.252.143,DHA,31,2,False,26,3,1
2,20790633968691748718,2025-03-26,146.71.50.198,DHA,4587,2,False,26,3,1
3,20790633968691748718,2025-03-26,191.96.36.115,DHA,3,2,False,26,3,1
4,20790633968691748718,2025-03-26,193.32.162.136,DHA,3,2,False,26,3,1
...,...,...,...,...,...,...,...,...,...,...
60419,20790633968691748718,2025-06-16,134.122.115.198,DHA,0,0,False,16,6,0
60420,20790633968691748718,2025-06-16,165.227.34.246,DHA,0,0,False,16,6,0
60421,20790633968691748718,2025-06-16,185.227.152.113,DHA,0,0,False,16,6,0
60422,20790633968691748718,2025-06-16,204.188.208.138,DHA,0,0,False,16,6,0


In [121]:
# Replace 'VA' with the correct OpDiv if needed
opdiv_merged['VA'][opdiv_merged['VA']['indicator'] == '192.124.249.112']

,API_UserName,date,indicator,OpDiv,observations,dayofweek,is_weekend,day,month,seen
144,74198686107399967946,2025-03-21,192.124.249.112,VA,0,4,False,21,3,0
814,74198686107399967946,2025-03-22,192.124.249.112,VA,0,5,True,22,3,0
1484,74198686107399967946,2025-03-23,192.124.249.112,VA,0,6,True,23,3,0
2154,74198686107399967946,2025-03-24,192.124.249.112,VA,0,0,False,24,3,0
2824,74198686107399967946,2025-03-25,192.124.249.112,VA,0,1,False,25,3,0
...,...,...,...,...,...,...,...,...,...,...
55754,74198686107399967946,2025-06-12,192.124.249.112,VA,39,3,False,12,6,1
56424,74198686107399967946,2025-06-13,192.124.249.112,VA,38,4,False,13,6,1
57094,74198686107399967946,2025-06-14,192.124.249.112,VA,21,5,True,14,6,1
57764,74198686107399967946,2025-06-15,192.124.249.112,VA,4,6,True,15,6,1


In [ ]:


def load_data(filepath, n_days=100):
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['date'])
    df.sort_values(by=['indicator', 'date'], inplace=True)
    latest_dates = df['date'].drop_duplicates().sort_values().tail(n_days)
    return df[df['date'].isin(latest_dates)].copy()

def extract_time_series_features(group):
    series = group['seen'].values
    indices = np.where(series == 1)[0]
    if len(indices) == 0:
        return pd.Series({
            'last_seen': len(series),
            'freq_7': 0,
            'freq_30': 0,
            'avg_gap': len(series),
            'burstiness': 0,
            'label_7': 0,
            'label_14': 0,
            'label_30': 0
        })
    last_seen = len(series) - 1 - indices[-1]
    freq_7 = np.sum(series[-7:])
    freq_30 = np.sum(series[-30:])
    gaps = np.diff(indices)
    avg_gap = np.mean(gaps) if len(gaps) > 0 else len(series)
    burstiness = (np.std(gaps) - avg_gap) / (np.std(gaps) + avg_gap) if len(gaps) > 1 else 0
    label_7 = 1 if np.any(series[-7:]) else 0
    label_14 = 1 if np.any(series[-14:]) else 0
    label_30 = 1 if np.any(series[-30:]) else 0
    return pd.Series({
        'last_seen': last_seen,
        'freq_7': freq_7,
        'freq_30': freq_30,
        'avg_gap': avg_gap,
        'burstiness': burstiness,
        'label_7': label_7,
        'label_14': label_14,
        'label_30': label_30
    })

def build_features(df):
    features_df = df.groupby('indicator').apply(extract_time_series_features).reset_index()
    return features_df

def train_predict(model_cls, X, y):
    if len(np.unique(y)) < 2:
        return np.full(len(y), np.nan)
    model = Pipeline([('scaler', StandardScaler()), ('clf', model_cls())])
    model.fit(X, y)
    return model.predict_proba(X)[:, 1]

def train_gbt(X, y):
    if len(np.unique(y)) < 2:
        return np.full(len(y), np.nan)
    model = GradientBoostingClassifier()
    model.fit(X, y)
    return model.predict_proba(X)[:, 1]

def fit_weibull_aft(X, avg_gap, event):
    aft_df = X.copy()
    aft_df['duration'] = avg_gap
    aft_df['event'] = event
    aft = WeibullAFTFitter()
    aft.fit(aft_df, duration_col='duration', event_col='event')
    return aft

def get_model_outputs(features_df, df):
    df_pred = features_df.copy()
    X = df_pred[['last_seen', 'freq_7', 'freq_30', 'avg_gap', 'burstiness']]
    y_7 = df_pred['label_7']
    y_14 = df_pred['label_14']
    y_30 = df_pred['label_30']

    # Logistic Regression, Estimate baseline probabilities
    df_pred['logistic_7'] = train_predict(LogisticRegression, X, y_7)
    df_pred['logistic_14'] = train_predict(LogisticRegression, X, y_14)
    df_pred['logistic_30'] = train_predict(LogisticRegression, X, y_30)

    # Gradient Boosted Trees, Non-linear patterns and feature interactions
    df_pred['gbt_7'] = train_gbt(X, y_7)
    df_pred['gbt_14'] = train_gbt(X, y_14)
    df_pred['gbt_30'] = train_gbt(X, y_30)

    # Exponential Model, memoryless Poisson process for frequency
    rate = (df_pred['freq_30'] / 30).clip(lower=1e-6)
    df_pred['exp_7'] = 1 - np.exp(-rate * 7)
    df_pred['exp_14'] = 1 - np.exp(-rate * 14)
    df_pred['exp_30'] = 1 - np.exp(-rate * 30)
    df_pred['exp_today'] = 1 - np.exp(-rate * 1)

    # Weibull AFT Model, time-to-event behavor, burstiness
    aft = fit_weibull_aft(X, df_pred['avg_gap'], y_7)
    surv_func = aft.predict_survival_function(X.assign(duration=df_pred['avg_gap'], event=y_7), times=[1, 7, 14, 30])
    df_pred['weibull_today'] = 1 - surv_func.loc[1].values
    df_pred['weibull_7'] = 1 - surv_func.loc[7].values
    df_pred['weibull_14'] = 1 - surv_func.loc[14].values
    df_pred['weibull_30'] = 1 - surv_func.loc[30].values

    # Today's forecast
    df_pred['logistic_today'] = train_predict(LogisticRegression, X, y_7)
    df_pred['gbt_today'] = train_gbt(X, y_7)

    # Merge in actual "seen" value for today's date
    latest_date = df['date'].max()
    today_seen = df[df['date'] == latest_date][['indicator', 'seen']].rename(columns={'seen': 'seen_today'})
    df_pred = df_pred.merge(today_seen, on='indicator', how='left')

    return df_pred

def add_rule_and_ensemble(output):
    features = ['last_seen', 'freq_7', 'freq_30', 'avg_gap', 'burstiness']
    X = output[features]

    # Rule-based labels
    output['rule_today'] = output['last_seen'].apply(lambda x: 1 if x == 0 else 0)
    output['rule_7d'] = output['last_seen'].apply(lambda x: 1 if x <= 6 else 0)
    output['rule_14d'] = output['last_seen'].apply(lambda x: 1 if x <= 13 else 0)
    output['rule_30d'] = output['last_seen'].apply(lambda x: 1 if x <= 29 else 0)

    y_today = output['rule_today']
    y_7 = output['rule_7d']
    y_14 = output['rule_14d']
    y_30 = output['rule_30d']

    def train_logistic_model(X, y):
        if len(np.unique(y)) < 2:
            return np.full(len(y), np.nan)
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression())
        ])
        model.fit(X, y)
        return model.predict_proba(X)[:, 1]

    output['prob_today'] = train_logistic_model(X, y_today)
    output['prob_7d'] = train_logistic_model(X, y_7)
    output['prob_14d'] = train_logistic_model(X, y_14)
    output['prob_30d'] = train_logistic_model(X, y_30)

    # Ensemble, combines predictions from all models and prevents overfitting
    output['ensemble_7d'] = (
        0.3 * output['prob_7d'].astype(float) +
        0.25 * output['gbt_7'] +
        0.25 * output['weibull_7'] +
        0.3 * output['exp_7']
    )
    output['ensemble_14d'] = (
        0.3 * output['prob_14d'].astype(float) +
        0.25 * output['gbt_14'] +
        0.25 * output['weibull_14'] +
        0.3 * output['exp_14']
    )
    output['ensemble_30d'] = (
        0.3 * output['prob_30d'].astype(float) +
        0.25 * output['gbt_30'] +
        0.25 * output['weibull_30'] +
        0.3 * output['exp_30']
    )
    return output

def classify_window(prob, freq, high_thresh, label):
    if prob >= high_thresh and freq >= 2:
        return f"{label}: Highly likely"
    elif prob >= 0.07 and freq >= 1:
        return f"{label}: Possibly active"
    else:
        return f"{label}: Low confidence"

def add_confidence_and_format(output):
    output['confidence_7d'] = output.apply(
        lambda row: classify_window(float(row['ensemble_7d']), row['freq_7'], 0.6, '7-Day'), axis=1
    )
    output['confidence_14d'] = output.apply(
        lambda row: classify_window(float(row['ensemble_14d']), row['freq_14'], 0.6, '14-Day'), axis=1
    )
    output['confidence_30d'] = output.apply(
        lambda row: classify_window(float(row['ensemble_30d']), row['freq_30'], 0.6, '30-Day'), axis=1
    )

    # Format percentages
    for col in ['prob_7d', 'prob_14d', 'prob_30d', 'ensemble_7d', 'ensemble_14d', 'ensemble_30d']:
        output[col] = np.clip(output[col].astype(float) * 100, 0, 100).round(2).astype(str) + '%'
    return output

def build_production_output(output):
    warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
    production_output = output[[
        'indicator', 'seen_today', 'freq_7', 'freq_30',
        'ensemble_7d', 'confidence_7d',
        'ensemble_14d', 'confidence_14d',
        'ensemble_30d', 'confidence_30d'
    ]].copy()
    production_output.rename(columns={
        'indicator': 'Indicator',
        'seen_today': 'Observed Today',
        'freq_7': 'Frequency (7d)',
        'freq_30': 'Frequency (30d)',
        'ensemble_7d': 'Probability: 7-Day',
        'confidence_7d': 'Confidence: 7-Day',
        'ensemble_14d': 'Probability: 14-Day',
        'confidence_14d': 'Confidence: 14-Day',
        'ensemble_30d': 'Probability: 30-Day',
        'confidence_30d': 'Confidence: 30-Day'
    }, inplace=True)
    return production_output


In [123]:
import os
import pandas as pd

def update_horizontal_forecast_log(production_df, actuals_df, log_csv_path):
    today = actuals_df['date'].max().normalize()
    col_name = f"Forecast Result: {today.strftime('%Y-%m-%d')}"
    records = {}

    # Step 1: Build new forecast tags for today
    for h in [7, 14, 30]:
        tag_col = f'Confidence: {h}-Day'
        target_date = today + pd.Timedelta(days=h)
        target_str = target_date.strftime('%Y-%m-%d')

        actuals_on_target = actuals_df[actuals_df['date'] == target_date][['indicator', 'seen']]
        actuals_map = dict(zip(actuals_on_target['indicator'], actuals_on_target['seen']))

        for _, row in production_df.iterrows():
            indicator = row['Indicator']
            forecast_tag = row[tag_col]
            was_seen = actuals_map.get(indicator, None)

            if target_date > today:
                outcome = "Pending"
            elif was_seen == 1:
                outcome = "Seen"
            elif was_seen == 0:
                outcome = "Missed"
            else:
                outcome = "Unknown"

            full_tag = f"{forecast_tag} -> {target_str} -> {outcome}"
            records[indicator] = records.get(indicator, '') + (f" | {full_tag}" if indicator in records else full_tag)

    new_col_df = pd.DataFrame.from_dict(records, orient='index', columns=[col_name])
    new_col_df.index.name = 'Indicator'
    new_col_df.reset_index(inplace=True)

    # Step 2: Load and update prior log if it exists
    if os.path.exists(log_csv_path):
        existing = pd.read_csv(log_csv_path)

        def update_cell(cell, indicator):
            if not isinstance(cell, str) or '-> Pending' not in cell:
                return cell

            segments = cell.split(' | ')
            updated_segments = []

            for segment in segments:
                try:
                    parts = segment.split('->')
                    if len(parts) != 3:
                        updated_segments.append(segment)
                        continue

                    tag, target_str, status = [p.strip() for p in parts]
                    target_date = pd.to_datetime(target_str)

                    if target_date.date() == today.date() and status == 'Pending':
                        seen = actuals_df[
                            (actuals_df['indicator'] == indicator) &
                            (actuals_df['date'] == today)
                        ]['seen'].values
                        outcome = 'Seen' if len(seen) > 0 and seen[0] == 1 else 'Missed' if len(seen) > 0 else 'Unknown'
                        updated_segments.append(f"{tag} -> {target_str} -> {outcome}")
                    else:
                        updated_segments.append(segment)
                except:
                    updated_segments.append(segment)

            return ' | '.join(updated_segments)

        for col in existing.columns:
            if col.startswith("Forecast Result: "):
                existing[col] = existing.apply(lambda row: update_cell(row[col], row['Indicator']), axis=1)

        if col_name in existing.columns:
            existing.drop(columns=[col_name], inplace=True)

        merged = pd.merge(existing, new_col_df, on='Indicator', how='outer')
    else:
        merged = new_col_df

    # Step 3: Limit to most recent 30 forecast columns
    forecast_cols = sorted(
        [col for col in merged.columns if col.startswith("Forecast Result: ")],
        reverse=True
    )
    recent_cols = forecast_cols[:30]  # Keep most recent 30 days
    final_cols = ['Indicator'] + sorted(recent_cols)  # Put Indicator first, then sorted recent cols
    merged = merged[final_cols]

    return merged

In [124]:
import warnings

# Silence pandas groupby apply deprecation warning globally for this notebook
warnings.filterwarnings("ignore", category=DeprecationWarning, message="DataFrameGroupBy.apply operated on the grouping columns")

from datetime import datetime
import os

def main():
    today = datetime.today().date()

    prediction_path = r'Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions'
    log_path = r'Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs'

    opdiv_production_outputs = {}
    opdiv_forecast_logs = {}

    for opdiv_name, opdiv_df in opdiv_merged.items():
        opdiv_df = opdiv_df.copy()
        features_df = build_features(opdiv_df)
        output = get_model_outputs(features_df, opdiv_df)
        output = add_rule_and_ensemble(output)
        output = add_confidence_and_format(output)
        production_output = build_production_output(output)

        # Save production output
        opdiv_production_outputs[opdiv_name] = production_output

        # Define path for existing forecast log CSV
        opdiv_log_dir = os.path.join(log_path, opdiv_name)
        os.makedirs(opdiv_log_dir, exist_ok=True)
        log_csv_path = os.path.join(opdiv_log_dir, f"{opdiv_name}_forecast_log.csv")

        # Update the log (merge with previous if exists)
        forecast_log = update_horizontal_forecast_log(production_output, opdiv_df, log_csv_path)
        opdiv_forecast_logs[opdiv_name] = forecast_log

        # Save updated forecast log
        forecast_log.to_csv(log_csv_path, index=False)

        # Save today's prediction CSV
        opdiv_output_dir = os.path.join(prediction_path, opdiv_name)
        os.makedirs(opdiv_output_dir, exist_ok=True)
        output_csv_path = os.path.join(opdiv_output_dir, f"{opdiv_name}_output_{today.strftime('%Y%m%d')}.csv")
        production_output.to_csv(output_csv_path, index=False)

    # Explicit unpacking of key OpDivs (optional for clarity)
    DHA_output = opdiv_production_outputs.get("DHA")
    CDC_output = opdiv_production_outputs.get("CDC")
    FDA_output = opdiv_production_outputs.get("FDA")
    NIH_output = opdiv_production_outputs.get("NIH")
    VA_output = opdiv_production_outputs.get("VA")
    HRSA_output = opdiv_production_outputs.get("HRSA")
    IHS_output = opdiv_production_outputs.get("IHS")
    OS_output = opdiv_production_outputs.get("OS")
    CMS_output = opdiv_production_outputs.get("CMS")
    HHS_output = opdiv_production_outputs.get("HHS")

    DHA_log = opdiv_forecast_logs.get("DHA")
    CDC_log = opdiv_forecast_logs.get("CDC")
    FDA_log = opdiv_forecast_logs.get("FDA")
    NIH_log = opdiv_forecast_logs.get("NIH")
    VA_log = opdiv_forecast_logs.get("VA")
    HRSA_log = opdiv_forecast_logs.get("HRSA")
    IHS_log = opdiv_forecast_logs.get("IHS")
    OS_log = opdiv_forecast_logs.get("OS")
    CMS_log = opdiv_forecast_logs.get("CMS")
    HHS_log = opdiv_forecast_logs.get("HHS")

    return {
        "DHA_output": DHA_output, "CDC_output": CDC_output, "FDA_output": FDA_output,
        "NIH_output": NIH_output, "VA_output": VA_output, "HRSA_output": HRSA_output,
        "IHS_output": IHS_output, "OS_output": OS_output, "CMS_output": CMS_output,
        "HHS_output": HHS_output,
        "DHA_log": DHA_log, "CDC_log": CDC_log, "FDA_log": FDA_log,
        "NIH_log": NIH_log, "VA_log": VA_log, "HRSA_log": HRSA_log,
        "IHS_log": IHS_log, "OS_log": OS_log, "CMS_log": CMS_log,
        "HHS_log": HHS_log
    }


if __name__ == "__main__":
    main()